# Add Entra ID Groups to Workspaces (Semantic Link)
Uses `sempy.fabric` for seamless Fabric authentication and workspace retrieval.  
**`DRY_RUN = True`** by default — flip to `False` to apply changes.

In [ ]:
import sempy.fabric as fabric
import pandas as pd
from datetime import datetime

# ============================================================================
# PARAMETERS
# ============================================================================

GROUPS_TO_ADD = [
    {"group_id": "aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee", "display_name": "PBI-Workspace-Viewers",      "role": "Viewer"},
    # {"group_id": "11111111-2222-3333-4444-555555555555", "display_name": "PBI-Workspace-Contributors", "role": "Contributor"},
]

# Target workspaces — pick ONE method:
TARGET_WORKSPACE_IDS  = []       # Option A: explicit string GUIDs
TARGET_ALL_WORKSPACES = False     # Option B: all non-personal workspaces
WORKSPACE_NAME_FILTER = ""       # Option C: substring match (e.g. "Finance")

DRY_RUN = True                    # True = preview only, no changes

print(f"Groups: {len(GROUPS_TO_ADD)} | All WS: {TARGET_ALL_WORKSPACES} | Filter: '{WORKSPACE_NAME_FILTER}' | Dry run: {DRY_RUN}")

In [ ]:
# --- Resolve Workspaces via Semantic Link -------------------------------------
if TARGET_ALL_WORKSPACES or WORKSPACE_NAME_FILTER:
    # list_workspaces() handles pagination and auth automatically
    df_ws = fabric.list_workspaces()
    df_ws = df_ws[df_ws["Type"] != "PersonalGroup"]  # Skip personal workspaces
    if WORKSPACE_NAME_FILTER:
        df_ws = df_ws[df_ws["Name"].str.contains(WORKSPACE_NAME_FILTER, case=False, na=False)]
    workspaces = df_ws[["Id", "Name"]].to_dict("records")

elif TARGET_WORKSPACE_IDS:
    # semantic link doesn't have a bulk 'get workspaces by ID' so we try to resolve names
    try:
        df_ws = fabric.list_workspaces()
        df_target = df_ws[df_ws["Id"].isin(TARGET_WORKSPACE_IDS)]
        workspaces = df_target[["Id", "Name"]].to_dict("records")
        # Add any unresolved IDs with a fallback name
        resolved_ids = set(df_target["Id"])
        for wid in TARGET_WORKSPACE_IDS:
            if wid not in resolved_ids:
                workspaces.append({"Id": wid, "Name": "(unresolved name)"})
    except:
        # Fallback if list_workspaces fails for some reason
        workspaces = [{"Id": wid, "Name": "Unknown"} for wid in TARGET_WORKSPACE_IDS]
else:
    workspaces = []
    print("⚠ No workspaces targeted.")

print(f"🎯 {len(workspaces)} workspace(s) targeted")
for w in workspaces[:10]: 
    print(f"   • {w['Name']}")
if len(workspaces) > 10:
    print(f"   ... +{len(workspaces)-10} more")

In [ ]:
# --- Add Groups via FabricRestClient ------------------------------------------
results = []
client = fabric.FabricRestClient() # Uses Semantic Link's authenticated client

print(f"\n{'🔍 DRY RUN' if DRY_RUN else '🚀 APPLYING'} — {datetime.now():%Y-%m-%d %H:%M}\n{'='*70}")

for ws in workspaces:
    print(f"\n📂 {ws['Name']}")
    for g in GROUPS_TO_ADD:
        if DRY_RUN:
            status, msg = "DRY_RUN", f"Would add '{g['display_name']}' as {g['role']}"
        else:
            try:
                # Power BI REST API (v1.0)
                url = f"v1.0/myorg/groups/{ws['Id']}/users"
                payload = {
                    "identifier": g["group_id"], 
                    "groupUserAccessRight": g["role"],
                    "principalType": "Group", 
                    "displayName": g["display_name"]
                }
                response = client.post(url, json=payload)
                
                if response.status_code == 200:
                    status, msg = "SUCCESS", f"Added '{g['display_name']}' as {g['role']}"
                elif "already" in response.text.lower():
                    status, msg = "SKIPPED", f"'{g['display_name']}' already has access"
                else:
                    status, msg = "ERROR", f"{response.status_code}: {response.text[:200]}"
            except Exception as e:
                status, msg = "ERROR", str(e)
        
        icon = {"SUCCESS":"✅","DRY_RUN":"🔍","SKIPPED":"⏭","ERROR":"❌"}.get(status,"❓")
        print(f"   {icon} {msg}")
        results.append({"Workspace": ws["Name"], "Group": g["display_name"], "Role": g["role"], "Status": status, "Detail": msg})

# --- Summary ------------------------------------------------------------------
df = pd.DataFrame(results)
print(f"\n{'='*70}")
if len(df) > 0:
    print(f"✅ {len(df[df['Status'].isin(['SUCCESS','DRY_RUN'])])} ok  |  ⏭ {len(df[df['Status']=='SKIPPED'])} skipped  |  ❌ {len(df[df['Status']=='ERROR'])} errors")
    if DRY_RUN: print("\n⚠ DRY RUN — set DRY_RUN = False and re-run to apply.")
    display(df)